## Cell 1 - English text

English text LM Studio English text, OpenAI/Gemini English text,English text ModelRegistry, Planner, Optimizer, JudgePool, PairwiseBattle, SelfCorrector.


In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

import openai

from src.core.model_provider import ModelRegistry
from src.core.providers.google_provider import GoogleProvider
from src.core.providers.lmstudio_provider import LMStudioProvider
from src.core.providers.openai_provider import OpenAIProvider
from src.core.token_estimator import TokenEstimator
from src.core.types import OptimizationConstraints
from src.evaluation.judge_pool import JudgePool
from src.evaluation.pairwise_battle import PairwiseBattle
from src.evaluation.self_correction import SelfCorrector
from src.optimization.optimizer import Optimizer
from src.planning.planner import Planner

LM_STUDIO_URL = os.getenv("LM_STUDIO_URL", "http://localhost:1234/v1").rstrip("/")
te = TokenEstimator()
e2e_rows = []
e2e_artifacts = {}
constraints = OptimizationConstraints(
    maxCompressionRate=0.50,
    minQualityScore=6.0,
    requirePositiveROI=True,
    maxSelfCorrectionRetries=2,
)

available_models = []
try:
    lmstudio_client = openai.OpenAI(base_url=LM_STUDIO_URL, api_key="lm-studio")
    available_models = [model.id for model in lmstudio_client.models.list().data]
except Exception as exc:
    raise RuntimeError(
        f"English text LM Studio local server: {type(exc).__name__}: {exc}. "
        "English text lms server start English text /v1/models English text."
    ) from exc

if not available_models:
    raise RuntimeError("LM Studio local server English text,English text /v1/models English text.")

MODEL_ID = available_models[0]
print("English text LM Studio English text:", available_models)
print("English text:", MODEL_ID)

lmstudio_provider = LMStudioProvider(model=MODEL_ID, base_url=LM_STUDIO_URL)
openai_provider = OpenAIProvider(model="gpt-4o")
google_provider = GoogleProvider(model="gemini-2.5-flash")

registry = ModelRegistry(
    planning_model=lmstudio_provider,
    optimization_model=lmstudio_provider,
    evaluation_models=[openai_provider, google_provider],
    tokenizer=te,
)
planner = Planner(
    model_provider=registry.planning_model,
    refs_dir=str(PROJECT_ROOT / "src/planning/instruction_refs"),
)
optimizer = Optimizer(
    model_provider=registry.optimization_model,
    token_estimator=registry.tokenizer,
    constraints=constraints,
)
judge_pool = JudgePool(registry.evaluation_models)
battle = PairwiseBattle(
    target_providers=registry.evaluation_models,
    judge_pool=judge_pool,
    constraints=constraints,
)
corrector = SelfCorrector(
    optimizer=optimizer,
    battle=battle,
    constraints=constraints,
)

print("ModelRegistry / Planner / Optimizer / PairwiseBattle / SelfCorrector English text.")


English text LM Studio English text: ['gemma-4-e4b-it', 'gpt-oss-nano', 'qwen3.5-4b', 'qwen3.5-9b-uncensored-hauhaucs-aggressive', 'qwen3.5-9b-abliterated', 'text-embedding-nomic-embed-text-v1.5']
English text: gemma-4-e4b-it


ModelRegistry / Planner / Optimizer / PairwiseBattle / SelfCorrector English text.


## Cell 2 - E2E English text 1:English text

English text React + TypeScript English text,English text Planning -> Optimization -> PairwiseBattle,English text.


In [2]:
print("=== E2E-01: English text ===")
raw_prompt_01 = "English text React + TypeScript English text,English text"

planning_01 = await planner.plan(raw_prompt_01)
optimization_01 = await optimizer.optimize(raw_prompt_01, planning_01)
evaluation_01 = await battle.evaluate(
    original_prompt=raw_prompt_01,
    optimized_prompt=optimization_01.optimizedPrompt,
    task_description=planning_01.detectedIntent,
)
final_prompt_01 = optimization_01.optimizedPrompt if evaluation_01.winner == "optimized" else raw_prompt_01

missing_summary_01 = [f"{field.importance}:{field.field}" for field in planning_01.missingFields]
print("Planning.scene:", planning_01.scene)
print("Planning.detectedIntent:", planning_01.detectedIntent)
print("English text:", missing_summary_01 or "English text")
print("English text prompt:")
print(optimization_01.optimizedPrompt)
print("English text:", f"{optimization_01.tokenStats.reductionRate:.1%}")
print("ROI report:", optimization_01.roiReport)
print("winner:", evaluation_01.winner)
print("overall score:", f"{evaluation_01.scores.overall:.2f}")

for vote in evaluation_01.judgeResults:
    print(f"JudgeVote: {vote.model} -> {vote.winner}")

e2e_artifacts["E2E-01"] = {
    "raw_prompt": raw_prompt_01,
    "planning": planning_01,
    "optimization": optimization_01,
    "evaluation": evaluation_01,
    "final_prompt": final_prompt_01,
}
e2e_rows.append({
    "English text": "E2E-01",
    "English text tokens": optimization_01.tokenStats.originalCount,
    "English text tokens": optimization_01.tokenStats.optimizedCount,
    "English text tokens": optimization_01.tokenStats.originalCount - optimization_01.tokenStats.optimizedCount,
    "English text winner": evaluation_01.winner,
    "overall": round(evaluation_01.scores.overall, 2),
})


=== E2E-01: English text ===


Planning.scene: vibe_coding
Planning.detectedIntent: English text,English text.
English text: ['recommended:constraints', 'recommended:output_format', 'optional:code_style']
English text prompt:
English text React + TypeScript English text,English text
English text: 0.0%
ROI report: inputTokensSaved=0 optimizationCostTokens=200 netTokenSavings=-200 roiPositive=False
winner: original
overall score: 9.28
JudgeVote: openai:gpt-4o -> original
JudgeVote: google:gemini-2.5-flash -> original


## Cell 3 - E2E English text 2:English text

English text prompt,English text Planning English text critical English text,English text Optimization -> PairwiseBattle;English text prompt English text SelfCorrector,English text prompt.


In [3]:
print("=== E2E-02: English text prompt + English text ===")
raw_prompt_02 = "English text"

planning_02 = await planner.plan(raw_prompt_02)
critical_missing_02 = [field for field in planning_02.missingFields if field.importance == "critical"]
print("Planning.scene:", planning_02.scene)
print("Planning.detectedIntent:", planning_02.detectedIntent)
print("critical English text:")
for field in critical_missing_02:
    print(f"- Required {field.field}: {field.question}")

optimization_02 = await optimizer.optimize(raw_prompt_02, planning_02)
evaluation_02_initial = await battle.evaluate(
    original_prompt=raw_prompt_02,
    optimized_prompt=optimization_02.optimizedPrompt,
    task_description=planning_02.detectedIntent,
)

print("English text winner:", evaluation_02_initial.winner)
print("English text overall:", f"{evaluation_02_initial.scores.overall:.2f}")
print("English text feedback:", evaluation_02_initial.feedback)

if evaluation_02_initial.winner == "original":
    print("English text: optimized prompt English text PairwiseBattle English text original.")
    final_prompt_02, evaluation_02 = await corrector.correct(
        raw_prompt=raw_prompt_02,
        planning_result=planning_02,
        initial_result=evaluation_02_initial,
        task_description=planning_02.detectedIntent,
    )
else:
    print("English text: English text,optimized English text.")
    final_prompt_02 = optimization_02.optimizedPrompt
    evaluation_02 = evaluation_02_initial

print("English text winner:", evaluation_02.winner)
print("English text overall:", f"{evaluation_02.scores.overall:.2f}")
print("English text prompt:")
print(final_prompt_02)

e2e_artifacts["E2E-02"] = {
    "raw_prompt": raw_prompt_02,
    "planning": planning_02,
    "optimization": optimization_02,
    "evaluation": evaluation_02,
    "initial_evaluation": evaluation_02_initial,
    "final_prompt": final_prompt_02,
}
e2e_rows.append({
    "English text": "E2E-02",
    "English text tokens": optimization_02.tokenStats.originalCount,
    "English text tokens": te.exact_count(final_prompt_02),
    "English text tokens": optimization_02.tokenStats.originalCount - te.exact_count(final_prompt_02),
    "English text winner": evaluation_02.winner,
    "overall": round(evaluation_02.scores.overall, 2),
})


=== E2E-02: English text prompt + English text ===


Planning.scene: vibe_coding
Planning.detectedIntent: Write a function
critical English text:
- Required language: English text(English text Python, TypeScript, Go)？
- Required framework: English text(English text React, FastAPI, Express)？


English text winner: original
English text overall: 1.75
English text feedback: Overall score 1.8 below threshold 6.0
English text: optimized prompt English text PairwiseBattle English text original.


English text winner: original
English text overall: 2.20
English text prompt:
English text


## Cell 4 - Token English text

English text token English text, English text overall score.


In [4]:
print("=== Token English text ===")
e2e_report_df = pd.DataFrame(
    e2e_rows,
    columns=["English text", "English text tokens", "English text tokens", "English text tokens", "English text winner", "overall"],
)
display(e2e_report_df)


=== Token English text ===


,English text,English text tokens,English text tokens,English text tokens,English text winner,overall
0,E2E-01,24,24,0,original,9.28
1,E2E-02,3,3,0,original,2.20


## Cell 5 - English text

English text Phase 4 English text.


In [5]:
print("=== E2E English text ===")
assert e2e_artifacts["E2E-01"]["planning"].scene == "vibe_coding"
assert e2e_artifacts["E2E-01"]["optimization"].roiReport is not None
assert any(
    field.importance == "critical"
    for field in e2e_artifacts["E2E-02"]["planning"].missingFields
)
assert e2e_artifacts["E2E-01"]["evaluation"].winner is not None
assert e2e_artifacts["E2E-02"]["evaluation"].winner is not None
print("=== E2E English text ===")


=== E2E English text ===
=== E2E English text ===
